# Triple exponential moving average crossover

A directional trend backtest based on fast and slow triple exponential moving averages, with benchmark comparison and transaction costs.

Daily OHLCV is downloaded from Yahoo Finance when the notebook runs. Open this notebook from the `notebooks/` directory so it can import the shared data loader.


In [ ]:
"""
TEMA Trading Strategy - Enhanced with Additional Metrics
========================================================
This implementation strictly follows the specified trading flow:
- Signal at Close[t]
- Execute at Open[t+1]  
- Value portfolio at Open[t+1]
- Returns = Open[t+2]/Open[t+1] - 1

Enhanced Features:
1. Rolling Beta calculation vs benchmark
2. Detailed monthly/yearly returns heatmap
3. Dollar value distributions and statistics
4. Enhanced performance metrics
"""

from _market_data import download_ohlcv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from typing import Dict, Tuple, Optional, List
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION - All parameters are configurable
# ============================================================================

class Config:
    """
    Centralized configuration for all strategy parameters.
    No hardcoded values throughout the code.
    """
    # API Configuration
    
    # Trading Parameters
    TICKER = 'TSLA'
    BENCHMARK = 'URTH'
    
    # Strategy Parameters  
    TEMA_PERIOD = 20  # TEMA lookback period in days
    
    # Backtest Period - 5 years ending 2018-12-31
    START_DATE = '2013-01-01'  # Start date for backtest
    END_DATE = '2018-12-31'    # End date for backtest
    
    # Transaction Costs - Real market conditions
    COMMISSION_RATE = 0.001    # 0.1% per trade (10 basis points)
    SLIPPAGE_RATE = 0.0005     # 0.05% slippage (5 basis points)
    
    # Capital
    INITIAL_CAPITAL = 100000   # Starting capital in USD
    
    # Rolling Window Parameters
    ROLLING_WINDOW_BETA = 126  # 6 months for rolling beta
    ROLLING_WINDOW_SHARPE = 252  # 1 year for rolling Sharpe
    ROLLING_WINDOW_VOL = 63   # 3 months for rolling volatility
    
    # Display Settings
    FIGURE_SIZE = (20, 28)
    DPI = 100
    
    # Color scheme
    COLORS = {
        'positive': '#27AE60',
        'negative': '#E74C3C',
        'neutral': '#95A5A6',
        'strategy': '#2E86AB',
        'benchmark': '#A23B72',
        'highlight': '#F39C12'
    }

# ============================================================================
# DATA ACQUISITION
# ============================================================================

def fetch_ohlc_data(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    try:
        frame = download_ohlcv(symbol, start_date, end_date)
    except RuntimeError as exc:
        print(exc)
        return pd.DataFrame()
    return frame[["Open", "Close"]].rename_axis("Date")

def validate_data(df: pd.DataFrame, ticker: str) -> bool:
    """
    Validate data integrity without any forward/backward filling.
    """
    
    print(f"\nData Validation for {ticker}:")
    print("-" * 40)
    
    # Check minimum data points
    min_points = 252  # At least 1 year
    if len(df) < min_points:
        print(f"✗ Insufficient data: {len(df)} points (need {min_points})")
        return False
    
    # Check and remove missing values - NO FILLING
    missing_before = df.isnull().sum().sum()
    if missing_before > 0:
        print(f"⚠ Found {missing_before} missing values - removing rows")
        df.dropna(inplace=True)
        print(f"  Remaining rows: {len(df)}")
    
    # Check for invalid prices
    invalid_mask = (df['Open'] <= 0) | (df['Close'] <= 0)
    if invalid_mask.any():
        print(f"⚠ Found {invalid_mask.sum()} invalid prices - removing")
        df = df[~invalid_mask]
    
    # Check for large date gaps
    date_diffs = df.index.to_series().diff()
    large_gaps = date_diffs[date_diffs > pd.Timedelta(days=10)]
    if len(large_gaps) > 0:
        print(f"⚠ Found {len(large_gaps)} large gaps (>10 days)")
        for date, gap in large_gaps.head(3).items():
            print(f"  {date.date()}: {gap.days} day gap")
    
    print(f"✓ Validation complete: {len(df)} valid trading days")
    return True

# ============================================================================
# TEMA CALCULATION
# ============================================================================

def calculate_tema(prices: pd.Series, period: int) -> pd.Series:
    """
    Calculate Triple Exponential Moving Average (TEMA).
    
    TEMA = 3 * EMA1 - 3 * EMA2 + EMA3
    
    Critical: No look-ahead bias - each value uses only historical data
    """
    
    # Calculate three sequential EMAs
    ema1 = prices.ewm(span=period, adjust=False).mean()
    ema2 = ema1.ewm(span=period, adjust=False).mean()
    ema3 = ema2.ewm(span=period, adjust=False).mean()
    
    # TEMA formula
    tema = 3 * ema1 - 3 * ema2 + ema3
    
    return tema

# ============================================================================
# SIGNAL GENERATION
# ============================================================================

def generate_signals(df: pd.DataFrame, tema_period: int) -> pd.DataFrame:
    """
    Generate trading signals based on TEMA direction.
    
    CRITICAL TIMING:
    - Signal generated at Close[t] based on TEMA[t] vs TEMA[t-1]
    - Position entered at Open[t+1]
    - No look-ahead bias
    """
    
    df = df.copy()
    
    # Calculate TEMA on Close prices
    df['TEMA'] = calculate_tema(df['Close'], tema_period)
    
    # TEMA change (today vs yesterday) - NO LOOK-AHEAD
    df['TEMA_Change'] = df['TEMA'].diff()
    
    # Generate signals based on TEMA direction
    df['Signal'] = 0  # Initialize
    
    # Only generate signals where TEMA is valid
    valid_mask = df['TEMA'].notna() & df['TEMA_Change'].notna()
    
    # Assign signals based on TEMA direction
    df.loc[valid_mask & (df['TEMA_Change'] > 0), 'Signal'] = 1   # Long
    df.loc[valid_mask & (df['TEMA_Change'] < 0), 'Signal'] = -1  # Short
    df.loc[valid_mask & (df['TEMA_Change'] == 0), 'Signal'] = 0  # Hold
    
    # Forward fill signals after first valid signal (always in market)
    first_valid = df[df['Signal'] != 0].index
    if len(first_valid) > 0:
        first_idx = first_valid[0]
        df.loc[first_idx:, 'Signal'] = df.loc[first_idx:, 'Signal'].replace(0, np.nan).fillna(method='ffill')
    
    # Fill remaining NaN with 0 (warmup period)
    df['Signal'] = df['Signal'].fillna(0)
    
    # CRITICAL: Shift signal by 1 day for execution at next Open
    df['Position'] = df['Signal'].shift(1).fillna(0)
    
    # Track position changes for trade counting
    df['Position_Change'] = df['Position'].diff().fillna(0)
    
    # Count trades (position changes, not including initial entry)
    trades = (df['Position_Change'] != 0).sum() - 1  # Subtract initial entry
    trades = max(0, trades)  # Ensure non-negative
    
    print(f"\nSignal Generation Summary:")
    print(f"  Warmup period: {(df['Position'] == 0).sum()} days")
    print(f"  Long positions: {(df['Position'] == 1).sum()} days")
    print(f"  Short positions: {(df['Position'] == -1).sum()} days")
    print(f"  Number of trades: {trades}")
    
    return df

# ============================================================================
# RETURN CALCULATION
# ============================================================================

def calculate_returns(df: pd.DataFrame, config: Config) -> pd.DataFrame:
    """
    Calculate returns following the EXACT specified methodology.
    
    CRITICAL FORMULA:
    - Entry at Open[t+1] after signal at Close[t]
    - Exit at Open[t+2]
    - Return = Open[t+2] / Open[t+1] - 1
    """
    
    df = df.copy()
    
    # CORRECT: Open-to-Open returns
    df['Market_Return'] = df['Open'].shift(-1) / df['Open'] - 1
    
    # Handle last row (no next day)
    df['Market_Return'] = df['Market_Return'].fillna(0)
    
    # Strategy return = Position * Market_Return
    df['Raw_Return'] = df['Position'] * df['Market_Return']
    
    # Transaction costs on position changes
    df['Transaction_Cost'] = 0.0
    
    # Apply commission and slippage on trades
    trade_mask = df['Position_Change'] != 0
    df.loc[trade_mask, 'Transaction_Cost'] = (
        abs(df.loc[trade_mask, 'Position_Change']) * 
        (config.COMMISSION_RATE + config.SLIPPAGE_RATE)
    )
    
    # Net strategy return after costs
    df['Strategy_Return'] = df['Raw_Return'] - df['Transaction_Cost']
    
    # Cumulative returns (compounded)
    df['Cumulative_Strategy'] = (1 + df['Strategy_Return']).cumprod()
    df['Cumulative_Market'] = (1 + df['Market_Return']).cumprod()
    
    # Portfolio value in dollars
    df['Portfolio_Value'] = config.INITIAL_CAPITAL * df['Cumulative_Strategy']
    
    # Daily P&L in dollars
    df['Daily_PnL'] = df['Portfolio_Value'].diff().fillna(0)
    
    # Calculate drawdown
    df['Running_Max'] = df['Portfolio_Value'].expanding().max()
    df['Drawdown'] = (df['Portfolio_Value'] - df['Running_Max']) / df['Running_Max']
    
    # Total transaction costs
    total_costs = df['Transaction_Cost'].sum() * config.INITIAL_CAPITAL
    
    print(f"\nReturn Calculation Summary:")
    print(f"  Total transaction costs: ${total_costs:.2f}")
    print(f"  Final portfolio value: ${df['Portfolio_Value'].iloc[-1]:,.2f}")
    print(f"  Total return: {(df['Cumulative_Strategy'].iloc[-1] - 1)*100:.2f}%")
    
    return df

# ============================================================================
# BENCHMARK COMPARISON & BETA CALCULATION
# ============================================================================

def calculate_benchmark_metrics(strategy_df: pd.DataFrame, benchmark_df: pd.DataFrame, 
                                config: Config) -> pd.DataFrame:
    """
    Calculate benchmark returns and rolling beta.
    """
    
    # Align dates
    common_dates = strategy_df.index.intersection(benchmark_df.index)
    
    if len(common_dates) == 0:
        print("Warning: No common dates between strategy and benchmark")
        return strategy_df
    
    # Create aligned dataframes
    aligned_df = strategy_df.loc[common_dates].copy()
    benchmark_aligned = benchmark_df.loc[common_dates].copy()
    
    # Calculate benchmark returns (same methodology)
    benchmark_aligned['Benchmark_Return'] = (
        benchmark_aligned['Open'].shift(-1) / benchmark_aligned['Open'] - 1
    )
    benchmark_aligned['Benchmark_Return'] = benchmark_aligned['Benchmark_Return'].fillna(0)
    
    # Add benchmark data to main dataframe
    aligned_df['Benchmark_Return'] = benchmark_aligned['Benchmark_Return']
    aligned_df['Cumulative_Benchmark'] = (1 + aligned_df['Benchmark_Return']).cumprod()
    
    # Calculate rolling beta
    window = config.ROLLING_WINDOW_BETA
    
    # Helper function to calculate beta
    def calc_beta(strategy_returns, benchmark_returns):
        if len(strategy_returns) < 20:  # Minimum points for meaningful beta
            return np.nan
        cov = np.cov(strategy_returns, benchmark_returns)[0, 1]
        var = np.var(benchmark_returns)
        return cov / var if var > 0 else np.nan
    
    # Calculate rolling beta
    aligned_df['Rolling_Beta'] = aligned_df['Strategy_Return'].rolling(window).apply(
        lambda x: calc_beta(
            x.values,
            aligned_df['Benchmark_Return'].iloc[-len(x):].values
        )
    )
    
    print(f"\nBenchmark Comparison:")
    print(f"  Benchmark: {config.BENCHMARK}")
    print(f"  Common trading days: {len(common_dates)}")
    print(f"  Latest rolling beta: {aligned_df['Rolling_Beta'].iloc[-1]:.3f}" 
          if not np.isnan(aligned_df['Rolling_Beta'].iloc[-1]) else "  Latest rolling beta: N/A")
    
    return aligned_df

# ============================================================================
# ENHANCED PERFORMANCE METRICS
# ============================================================================

def calculate_enhanced_metrics(df: pd.DataFrame, config: Config) -> Dict:
    """
    Calculate comprehensive performance metrics including dollar values.
    """
    
    # Clean returns for calculation
    returns = df['Strategy_Return'].dropna()
    
    # Time calculations
    trading_days = len(returns)
    years = trading_days / 252
    
    # Return metrics
    total_return = df['Cumulative_Strategy'].iloc[-1] - 1
    annual_return = (1 + total_return) ** (1/years) - 1 if years > 0 else 0
    
    # Dollar metrics
    initial_value = config.INITIAL_CAPITAL
    final_value = df['Portfolio_Value'].iloc[-1]
    total_profit = final_value - initial_value
    
    # Daily P&L statistics in dollars
    daily_pnl = df['Daily_PnL'].dropna()
    avg_daily_pnl = daily_pnl.mean()
    std_daily_pnl = daily_pnl.std()
    max_daily_gain = daily_pnl.max()
    max_daily_loss = daily_pnl.min()
    
    # Risk metrics
    volatility = returns.std() * np.sqrt(252)
    sharpe = annual_return / volatility if volatility > 0 else 0
    
    # Downside risk metrics
    downside_returns = returns[returns < 0]
    downside_dev = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else 0
    sortino = annual_return / downside_dev if downside_dev > 0 else 0
    
    # Drawdown metrics
    max_dd = df['Drawdown'].min()
    max_dd_dollars = abs(max_dd * df['Running_Max'].loc[df['Drawdown'].idxmin()])
    
    # Recovery metrics
    if max_dd < 0:
        dd_start = df['Drawdown'].idxmin()
        recovery_mask = (df.index > dd_start) & (df['Portfolio_Value'] >= df['Running_Max'].loc[dd_start])
        if recovery_mask.any():
            recovery_date = df.index[recovery_mask][0]
            recovery_days = (recovery_date - dd_start).days
        else:
            recovery_days = None
    else:
        recovery_days = None
    
    # Win rate and profit metrics
    winning_days = (returns > 0).sum()
    losing_days = (returns < 0).sum()
    win_rate = winning_days / len(returns) if len(returns) > 0 else 0
    
    winning_returns = returns[returns > 0]
    losing_returns = returns[returns < 0]
    
    avg_win = winning_returns.mean() if len(winning_returns) > 0 else 0
    avg_loss = losing_returns.mean() if len(losing_returns) > 0 else 0
    
    # Profit factor
    gross_profits = winning_returns.sum()
    gross_losses = abs(losing_returns.sum())
    profit_factor = gross_profits / gross_losses if gross_losses > 0 else 0
    
    # Trade statistics
    trades = (df['Position_Change'] != 0).sum() - 1
    trades = max(0, trades)
    
    # Beta and correlation metrics (if benchmark available)
    if 'Benchmark_Return' in df.columns:
        # Full period beta
        strategy_ret = df['Strategy_Return'].dropna()
        benchmark_ret = df['Benchmark_Return'].dropna()
        
        if len(strategy_ret) == len(benchmark_ret) and len(strategy_ret) > 0:
            correlation = strategy_ret.corr(benchmark_ret)
            covariance = strategy_ret.cov(benchmark_ret)
            benchmark_var = benchmark_ret.var()
            beta = covariance / benchmark_var if benchmark_var > 0 else 0
            
            # Alpha calculation
            risk_free = 0.02  # 2% annual risk-free rate
            benchmark_annual = (1 + benchmark_ret.sum()) ** (1/years) - 1 if years > 0 else 0
            alpha = annual_return - (risk_free + beta * (benchmark_annual - risk_free))
            
            # Information ratio
            excess_returns = strategy_ret - benchmark_ret
            tracking_error = excess_returns.std() * np.sqrt(252)
            info_ratio = (annual_return - benchmark_annual) / tracking_error if tracking_error > 0 else 0
        else:
            correlation = beta = alpha = info_ratio = np.nan
    else:
        correlation = beta = alpha = info_ratio = np.nan
    
    # Monthly returns statistics
    monthly_returns = df.set_index(df.index)['Strategy_Return'].resample('M').apply(
        lambda x: (1+x).prod()-1 if len(x) > 0 else np.nan
    ).dropna()
    
    if len(monthly_returns) > 0:
        best_month = monthly_returns.max()
        worst_month = monthly_returns.min()
        positive_months = (monthly_returns > 0).sum()
        negative_months = (monthly_returns < 0).sum()
        monthly_win_rate = positive_months / len(monthly_returns)
    else:
        best_month = worst_month = 0
        positive_months = negative_months = 0
        monthly_win_rate = 0
    
    return {
        # Return Metrics
        'Total Return': total_return,
        'Annual Return': annual_return,
        'Total Profit ($)': total_profit,
        'Final Portfolio Value ($)': final_value,
        
        # Risk Metrics
        'Volatility': volatility,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Max Drawdown': max_dd,
        'Max Drawdown ($)': max_dd_dollars,
        'Recovery Days': recovery_days if recovery_days else 'Still in drawdown',
        
        # Daily Statistics
        'Avg Daily P&L ($)': avg_daily_pnl,
        'Std Daily P&L ($)': std_daily_pnl,
        'Max Daily Gain ($)': max_daily_gain,
        'Max Daily Loss ($)': max_daily_loss,
        
        # Win/Loss Statistics
        'Win Rate': win_rate,
        'Winning Days': winning_days,
        'Losing Days': losing_days,
        'Average Win': avg_win,
        'Average Loss': avg_loss,
        'Profit Factor': profit_factor,
        
        # Monthly Statistics
        'Best Month': best_month,
        'Worst Month': worst_month,
        'Positive Months': positive_months,
        'Negative Months': negative_months,
        'Monthly Win Rate': monthly_win_rate,
        
        # Trading Statistics
        'Number of Trades': trades,
        'Trading Days': trading_days,
        
        # Benchmark Metrics (if available)
        'Beta': beta,
        'Alpha': alpha,
        'Correlation': correlation,
        'Information Ratio': info_ratio
    }

# ============================================================================
# ENHANCED VISUALIZATION
# ============================================================================

def create_enhanced_visualizations(df: pd.DataFrame, metrics: Dict, config: Config):
    """
    Create comprehensive performance visualization with enhanced metrics.
    """
    
    fig = plt.figure(figsize=config.FIGURE_SIZE, dpi=config.DPI)
    
    # Create grid for subplots
    gs = fig.add_gridspec(8, 3, hspace=0.3, wspace=0.3)
    
    # 1. Price and TEMA (full width)
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(df.index, df['Close'], label='Close Price', alpha=0.7, linewidth=1, color='black')
    ax1.plot(df.index, df['TEMA'], label=f'TEMA ({config.TEMA_PERIOD})', 
             color=config.COLORS['strategy'], linewidth=2)
    
    # Mark trade signals
    buy_signals = df[df['Position_Change'] > 0]
    sell_signals = df[df['Position_Change'] < 0]
    ax1.scatter(buy_signals.index, buy_signals['Close'], 
               color=config.COLORS['positive'], marker='^', s=50, alpha=0.7, label='Long Entry')
    ax1.scatter(sell_signals.index, sell_signals['Close'], 
               color=config.COLORS['negative'], marker='v', s=50, alpha=0.7, label='Short Entry')
    
    ax1.set_title(f'{config.TICKER} Price and TEMA Indicator', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Price ($)')
    ax1.legend(loc='best', fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # 2. Cumulative Returns (full width)
    ax2 = fig.add_subplot(gs[1, :])
    ax2.plot(df.index, (df['Cumulative_Strategy'] - 1) * 100, 
            label='Strategy', color=config.COLORS['strategy'], linewidth=2)
    ax2.plot(df.index, (df['Cumulative_Market'] - 1) * 100, 
            label=f'{config.TICKER} Buy & Hold', color='gray', alpha=0.7, linewidth=1.5)
    
    if 'Cumulative_Benchmark' in df.columns:
        ax2.plot(df.index, (df['Cumulative_Benchmark'] - 1) * 100, 
                label=f'{config.BENCHMARK} Benchmark', color=config.COLORS['benchmark'], 
                linewidth=1.5, linestyle='--')
    
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax2.set_title('Cumulative Returns Comparison', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Cumulative Return (%)')
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    
    # 3. Portfolio Value in Dollars
    ax3 = fig.add_subplot(gs[2, :])
    ax3.plot(df.index, df['Portfolio_Value'], 
            color=config.COLORS['strategy'], linewidth=2)
    ax3.axhline(y=config.INITIAL_CAPITAL, color='black', linestyle='--', 
               alpha=0.5, label=f'Initial: ${config.INITIAL_CAPITAL:,.0f}')
    
    # Add annotations for key values
    final_value = df['Portfolio_Value'].iloc[-1]
    ax3.annotate(f'Final: ${final_value:,.0f}', 
                xy=(df.index[-1], final_value),
                xytext=(10, 10), textcoords='offset points',
                fontsize=9, fontweight='bold',
                color=config.COLORS['positive'] if final_value > config.INITIAL_CAPITAL else config.COLORS['negative'])
    
    ax3.set_title('Portfolio Value Over Time', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Date')
    ax3.set_ylabel('Portfolio Value ($)')
    ax3.legend(loc='best')
    ax3.grid(True, alpha=0.3)
    
    # 4. Drawdown
    ax4 = fig.add_subplot(gs[3, :2])
    ax4.fill_between(df.index, df['Drawdown'] * 100, 0, 
                     color=config.COLORS['negative'], alpha=0.3)
    ax4.plot(df.index, df['Drawdown'] * 100, color=config.COLORS['negative'], linewidth=1)
    
    # Mark maximum drawdown
    max_dd_idx = df['Drawdown'].idxmin()
    ax4.scatter(max_dd_idx, df.loc[max_dd_idx, 'Drawdown'] * 100, 
               color=config.COLORS['negative'], s=100, zorder=5)
    ax4.annotate(f'Max DD: {df.loc[max_dd_idx, "Drawdown"]*100:.1f}%\n${metrics["Max Drawdown ($)"]:,.0f}',
                xy=(max_dd_idx, df.loc[max_dd_idx, 'Drawdown'] * 100),
                xytext=(10, -10), textcoords='offset points',
                fontsize=9, fontweight='bold',
                color=config.COLORS['negative'])
    
    ax4.set_title('Strategy Drawdown', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Date')
    ax4.set_ylabel('Drawdown (%)')
    ax4.grid(True, alpha=0.3)
    
    # 5. Daily P&L Distribution in Dollars
    ax5 = fig.add_subplot(gs[3, 2])
    daily_pnl = df['Daily_PnL'].dropna()
    
    # Create histogram with different colors for positive and negative
    n, bins, patches = ax5.hist(daily_pnl, bins=50, edgecolor='black', linewidth=0.5)
    
    # Color bars based on positive/negative
    for i, patch in enumerate(patches):
        if bins[i] >= 0:
            patch.set_facecolor(config.COLORS['positive'])
            patch.set_alpha(0.7)
        else:
            patch.set_facecolor(config.COLORS['negative'])
            patch.set_alpha(0.7)
    
    ax5.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax5.axvline(x=daily_pnl.mean(), color=config.COLORS['highlight'], 
               linestyle='--', linewidth=2, label=f'Mean: ${daily_pnl.mean():,.0f}')
    
    # Add text box with statistics
    stats_text = f'Mean: ${daily_pnl.mean():,.0f}\n'
    stats_text += f'Std: ${daily_pnl.std():,.0f}\n'
    stats_text += f'Max Gain: ${daily_pnl.max():,.0f}\n'
    stats_text += f'Max Loss: ${daily_pnl.min():,.0f}'
    
    ax5.text(0.02, 0.98, stats_text, transform=ax5.transAxes,
            fontsize=8, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    ax5.set_title('Daily P&L Distribution ($)', fontsize=12, fontweight='bold')
    ax5.set_xlabel('Daily P&L ($)')
    ax5.set_ylabel('Frequency')
    ax5.grid(True, alpha=0.3)
    
    # 6. Rolling Beta (if available)
    ax6 = fig.add_subplot(gs[4, :2])
    if 'Rolling_Beta' in df.columns:
        ax6.plot(df.index, df['Rolling_Beta'], 
                color=config.COLORS['benchmark'], linewidth=1.5)
        ax6.axhline(y=1, color='black', linestyle='--', alpha=0.5, linewidth=0.8)
        ax6.axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=0.5)
        
        # Add average beta line
        avg_beta = df['Rolling_Beta'].mean()
        if not np.isnan(avg_beta):
            ax6.axhline(y=avg_beta, color=config.COLORS['highlight'], 
                       linestyle='--', alpha=0.7, 
                       label=f'Avg Beta: {avg_beta:.2f}')
        
        ax6.set_title(f'Rolling Beta ({config.ROLLING_WINDOW_BETA}-day window)', 
                     fontsize=12, fontweight='bold')
        ax6.set_xlabel('Date')
        ax6.set_ylabel('Beta')
        ax6.legend(loc='best')
        ax6.grid(True, alpha=0.3)
    else:
        ax6.text(0.5, 0.5, 'Benchmark data not available', 
                ha='center', va='center', transform=ax6.transAxes)
        ax6.set_title('Rolling Beta', fontsize=12, fontweight='bold')
    
    # 7. Rolling Volatility
    ax7 = fig.add_subplot(gs[4, 2])
    rolling_vol = df['Strategy_Return'].rolling(config.ROLLING_WINDOW_VOL).std() * np.sqrt(252) * 100
    ax7.plot(df.index, rolling_vol, color=config.COLORS['neutral'], linewidth=1.5)
    
    # Add average volatility line
    avg_vol = rolling_vol.mean()
    if not np.isnan(avg_vol):
        ax7.axhline(y=avg_vol, color=config.COLORS['highlight'], 
                   linestyle='--', alpha=0.7, 
                   label=f'Avg Vol: {avg_vol:.1f}%')
    
    ax7.set_title(f'Rolling Volatility ({config.ROLLING_WINDOW_VOL}-day)', 
                 fontsize=12, fontweight='bold')
    ax7.set_xlabel('Date')
    ax7.set_ylabel('Annualized Volatility (%)')
    ax7.legend(loc='best')
    ax7.grid(True, alpha=0.3)
    
    # 8. Monthly Returns Heatmap
    ax8 = fig.add_subplot(gs[5:7, :2])
    
    # Calculate monthly returns
    monthly_returns = df.set_index(df.index)['Strategy_Return'].resample('M').apply(
        lambda x: (1+x).prod()-1 if len(x) > 0 else np.nan
    )
    
    if len(monthly_returns) > 0:
        # Create pivot table for heatmap
        monthly_pivot = monthly_returns.to_frame()
        monthly_pivot['Year'] = monthly_pivot.index.year
        monthly_pivot['Month'] = monthly_pivot.index.month
        monthly_pivot = monthly_pivot.pivot(index='Year', columns='Month', 
                                           values='Strategy_Return')
        
        # Create heatmap
        sns.heatmap(monthly_pivot * 100, annot=True, fmt='.1f', 
                   cmap='RdYlGn', center=0,
                   cbar_kws={'label': 'Monthly Return (%)'},
                   ax=ax8, vmin=-20, vmax=20,
                   mask=monthly_pivot.isna())
        
        # Customize month labels
        month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                       'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
        ax8.set_xticklabels([month_labels[i-1] for i in monthly_pivot.columns])
        
        ax8.set_title('Monthly Returns Heatmap (%)', fontsize=12, fontweight='bold')
        ax8.set_xlabel('Month')
        ax8.set_ylabel('Year')
    
    # 9. Yearly Returns Bar Chart
    ax9 = fig.add_subplot(gs[5:7, 2])
    
    # Calculate yearly returns
    yearly_returns = df.set_index(df.index)['Strategy_Return'].resample('Y').apply(
        lambda x: (1+x).prod()-1 if len(x) > 0 else np.nan
    ).dropna()
    
    if len(yearly_returns) > 0:
        years = yearly_returns.index.year
        colors = [config.COLORS['positive'] if r > 0 else config.COLORS['negative'] 
                 for r in yearly_returns]
        
        bars = ax9.bar(years, yearly_returns * 100, color=colors, 
                      edgecolor='black', linewidth=1, alpha=0.7)
        
        # Add value labels on bars
        for bar, value in zip(bars, yearly_returns * 100):
            height = bar.get_height()
            ax9.text(bar.get_x() + bar.get_width()/2., height,
                    f'{value:.1f}%',
                    ha='center', va='bottom' if height > 0 else 'top',
                    fontsize=9, fontweight='bold')
        
        ax9.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        ax9.set_title('Annual Returns', fontsize=12, fontweight='bold')
        ax9.set_xlabel('Year')
        ax9.set_ylabel('Annual Return (%)')
        ax9.grid(True, alpha=0.3, axis='y')
    
    # 10. Performance Metrics Table (enhanced with dollar values)
    ax10 = fig.add_subplot(gs[7, :])
    ax10.axis('tight')
    ax10.axis('off')
    
    # Prepare metrics for display - organized by category
    metrics_categories = [
        ('RETURNS', [
            ('Total Return', f"{metrics['Total Return']:.2%}"),
            ('Annual Return', f"{metrics['Annual Return']:.2%}"),
            ('Total Profit', f"${metrics['Total Profit ($)']:,.0f}"),
            ('Final Value', f"${metrics['Final Portfolio Value ($)']:,.0f}"),
        ]),
        ('RISK', [
            ('Volatility', f"{metrics['Volatility']:.2%}"),
            ('Sharpe Ratio', f"{metrics['Sharpe Ratio']:.3f}"),
            ('Sortino Ratio', f"{metrics['Sortino Ratio']:.3f}"),
            ('Max Drawdown', f"{metrics['Max Drawdown']:.2%} (${metrics['Max Drawdown ($)']:,.0f})"),
        ]),
        ('DAILY P&L', [
            ('Avg Daily P&L', f"${metrics['Avg Daily P&L ($)']:,.0f}"),
            ('Std Daily P&L', f"${metrics['Std Daily P&L ($)']:,.0f}"),
            ('Max Gain', f"${metrics['Max Daily Gain ($)']:,.0f}"),
            ('Max Loss', f"${metrics['Max Daily Loss ($)']:,.0f}"),
        ]),
        ('WIN/LOSS', [
            ('Win Rate', f"{metrics['Win Rate']:.2%}"),
            ('Profit Factor', f"{metrics['Profit Factor']:.3f}"),
            ('Avg Win', f"{metrics['Average Win']:.3%}"),
            ('Avg Loss', f"{metrics['Average Loss']:.3%}"),
        ]),
        ('BENCHMARK', [
            ('Beta', f"{metrics['Beta']:.3f}" if not np.isnan(metrics['Beta']) else 'N/A'),
            ('Alpha', f"{metrics['Alpha']:.3%}" if not np.isnan(metrics['Alpha']) else 'N/A'),
            ('Correlation', f"{metrics['Correlation']:.3f}" if not np.isnan(metrics['Correlation']) else 'N/A'),
            ('Info Ratio', f"{metrics['Information Ratio']:.3f}" if not np.isnan(metrics['Information Ratio']) else 'N/A'),
        ]),
    ]
    
    # Create multi-column table
    table_data = []
    max_rows = max(len(cat[1]) for cat in metrics_categories)
    
    for i in range(max_rows):
        row = []
        for category_name, category_metrics in metrics_categories:
            if i < len(category_metrics):
                row.extend(category_metrics[i])
            else:
                row.extend(['', ''])
        table_data.append(row)
    
    # Create headers
    col_labels = []
    for category_name, _ in metrics_categories:
        col_labels.extend([category_name, 'Value'])
    
    table = ax10.table(cellText=table_data,
                      colLabels=col_labels,
                      cellLoc='left',
                      loc='center',
                      colWidths=[0.15, 0.15] * len(metrics_categories))
    
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1.2, 1.5)
    
    # Style the headers
    for i in range(0, len(col_labels), 2):
        table[(0, i)].set_facecolor('#4CAF50')
        table[(0, i)].set_text_props(weight='bold', color='white')
        table[(0, i+1)].set_facecolor('#4CAF50')
        table[(0, i+1)].set_text_props(weight='bold', color='white')
    
    # Alternate row colors
    for i in range(1, len(table_data) + 1):
        for j in range(len(col_labels)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#f0f0f0')
    
    ax10.set_title('Comprehensive Performance Metrics', fontsize=14, fontweight='bold', pad=20)
    
    # Main title
    plt.suptitle(f'TEMA Trading Strategy - Enhanced Analysis\n'
                f'{config.TICKER} | Period: {df.index[0].date()} to {df.index[-1].date()} | '
                f'Initial Capital: ${config.INITIAL_CAPITAL:,.0f}',
                fontsize=16, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    plt.show()

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """
    Main execution function with enhanced metrics and visualizations.
    """
    
    print("="*60)
    print("TEMA TRADING STRATEGY - ENHANCED VERSION")
    print("="*60)
    
    # Initialize configuration
    config = Config()
    
    print(f"\nConfiguration:")
    print(f"  Ticker: {config.TICKER}")
    print(f"  Benchmark: {config.BENCHMARK}")
    print(f"  Period: {config.START_DATE} to {config.END_DATE}")
    print(f"  TEMA Period: {config.TEMA_PERIOD} days")
    print(f"  Initial Capital: ${config.INITIAL_CAPITAL:,.0f}")
    print(f"  Commission: {config.COMMISSION_RATE:.2%}")
    print(f"  Slippage: {config.SLIPPAGE_RATE:.2%}")
    
    # 1. Fetch ticker data
    print(f"\n{'='*40}")
    print("STEP 1: Fetching Ticker Data")
    print('='*40)
    
    df = fetch_ohlc_data(config.TICKER, config.START_DATE, config.END_DATE)
    
    if df.empty:
        print("ERROR: No data retrieved for ticker")
        return
    
    # 2. Validate data
    print(f"\n{'='*40}")
    print("STEP 2: Validating Data")
    print('='*40)
    
    if not validate_data(df, config.TICKER):
        print("ERROR: Data validation failed")
        return
    
    # 3. Generate signals
    print(f"\n{'='*40}")
    print("STEP 3: Generating Trading Signals")
    print('='*40)
    
    df = generate_signals(df, config.TEMA_PERIOD)
    
    # 4. Calculate returns
    print(f"\n{'='*40}")
    print("STEP 4: Calculating Returns & P&L")
    print('='*40)
    
    df = calculate_returns(df, config)
    
    # 5. Fetch and process benchmark
    print(f"\n{'='*40}")
    print("STEP 5: Fetching Benchmark Data")
    print('='*40)
    
    benchmark_df = fetch_ohlc_data(config.BENCHMARK, config.START_DATE, config.END_DATE)
    
    if not benchmark_df.empty:
        df = calculate_benchmark_metrics(df, benchmark_df, config)
    else:
        print(f"Warning: Could not fetch benchmark {config.BENCHMARK}")
    
    # 6. Calculate enhanced metrics
    print(f"\n{'='*40}")
    print("STEP 6: Computing Enhanced Metrics")
    print('='*40)
    
    metrics = calculate_enhanced_metrics(df, config)
    
    # Print summary
    print("\nPerformance Summary:")
    print("-"*40)
    print(f"Total Return: {metrics['Total Return']:.2%}")
    print(f"Total Profit: ${metrics['Total Profit ($)']:,.2f}")
    print(f"Final Portfolio Value: ${metrics['Final Portfolio Value ($)']:,.2f}")
    print(f"Sharpe Ratio: {metrics['Sharpe Ratio']:.3f}")
    print(f"Max Drawdown: {metrics['Max Drawdown']:.2%} (${metrics['Max Drawdown ($)']:,.0f})")
    print(f"Average Daily P&L: ${metrics['Avg Daily P&L ($)']:,.2f}")
    print(f"Win Rate: {metrics['Win Rate']:.2%}")
    
    if not np.isnan(metrics['Beta']):
        print(f"Beta: {metrics['Beta']:.3f}")
        print(f"Alpha: {metrics['Alpha']:.3%}")
    
    # 7. Create enhanced visualizations
    print(f"\n{'='*40}")
    print("STEP 7: Creating Enhanced Visualizations")
    print('='*40)
    
    create_enhanced_visualizations(df, metrics, config)
    
    print(f"\n{'='*40}")
    print("EXECUTION COMPLETE")
    print('='*40)
    
    return df, metrics

# Run the enhanced strategy
if __name__ == "__main__":
    results_df, performance_metrics = main()